# SAML-D v1 — prototyp cech sieciowych i historycznych

Cel: sprawdzić **przed** freeze i pipeline'em, czy cechy v1 rozdzielają pary, których v0 nie rozróżnia
(Layered_Fan_In vs Fan_In, Stacked Bipartite vs Bipartite, Cycle, Behavioural_Change_1 vs 2).

Protokół: pule **walidacyjne** foldów 1–3 (test nietknięty), te same epizody dla v0 i v1, konta rozłączne
między supportem a zapytaniami, standaryzacja tylko na klasach treningowych folda.

**Kryterium (ustalone przed uruchomieniem)**, średni recall sześciu słabych klas, ProtoNet v1 − ProtoNet v0 (w parach):
- Δ ≥ 0,10 → PASS: freeze `samld-v1`, podłogi, kampania.
- Δ < 0,10, ale `nc_raw`/`rf` na v1 rosną o ≥ 0,10 → informacja jest, ProtoNet ją tłumi (metryka uczona na klasach, gdzie ta cecha nic nie znaczyła) → poprawka w modelu (dystans hybrydowy), nie GNN.
- wzrost częściowy 0,05–0,10 → informacja niewyczerpana → uzasadnienie dla GNN.
- brak wzrostu → sygnatura nie należy do pojedynczego konta → klasyfikacja przypadków (grup transakcji).

Kernel: `fsl (py3.10)`. Cechy liczą się ok. 2–5 min, ewaluacja ok. 10–15 min (CPU).

In [ ]:
import sys, json, time, logging
sys.path.insert(0, "/home/jupyter/mlops-meta-learning/fsl-projekt/src")
from pathlib import Path
import numpy as np, pandas as pd, torch
from collections import defaultdict
from types import SimpleNamespace
from sklearn.ensemble import RandomForestClassifier
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.data.samld import GroupedEpisodeSampler
from fsl.models.mlp import MLP
from fsl.models.protonet import fast_adapt
logging.basicConfig(level=logging.ERROR)

SRC = "/home/jupyter/data_cache/saml_d_source/SAML-D.csv"
FEAT_CACHE = Path("/home/jupyter/data_cache/samld_v1_proto_features.parquet")
RESULTS = Path("/home/jupyter/data_cache/samld_v1_proto_results.json")

cfg0 = TrainConfig(project="dark-data-discovery", dataset="samld", seed=0, fold=0, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg0)
bundle, sha_v0 = load_frozen(cfg0)
print("v0:", sha_v0[:12], "| okien", len(bundle["windows"]), "| cech", len(bundle["schema"]["feature_names"]))

## Cechy v1 (bez etykiet; historia tylko wstecz)

In [ ]:
NEW = ["fwd_same", "fwd_1d", "bal_exact", "n_bal",
       "in_nb_fwd", "out_nb_fwd", "in_nb_indeg", "out_nb_outdeg",
       "cyc2", "cyc3",
       "hist_active4", "hist_n_lr", "hist_amt_lr", "acct_age", "new_cp_share",
       "max_day_n", "max_day_share"]


def build_v1_features(tx: pd.DataFrame, win: pd.DataFrame, cap: int = 30, log=print) -> pd.DataFrame:
    """tx: kolumny s, r, a, day, week (wszystkie transakcje).
    win: okna docelowe z kolumnami acc, week, n, n_out, amt_in_sum, amt_out_sum (z freeze v0).
    cap: konta z tygodniowym out-degree > cap nie sa srodkowym ogniwem cyklu (ochrona zlaczen).
    Zwraca DataFrame acc, week + NEW. Nic tu nie dotyka etykiet.
    """
    tgt = win[["acc", "week"]].drop_duplicates().reset_index(drop=True)
    tgt_acc = tgt["acc"].unique()

    # --- 1. dzienne wplywy/wyplywy kazdego konta (sasiedzi tez ich potrzebuja) ---
    o = (tx.groupby(["s", "day"], sort=False)
           .agg(out_amt=("a", "sum"), out_n=("a", "size")).reset_index().rename(columns={"s": "acc"}))
    i = (tx.groupby(["r", "day"], sort=False)
           .agg(in_amt=("a", "sum"), in_n=("a", "size")).reset_index().rename(columns={"r": "acc"}))
    ad = o.merge(i, on=["acc", "day"], how="outer").fillna(0.0)
    del o, i
    ad["week"] = (ad["day"] // 7).astype("int32")
    nxt = ad[["acc", "day", "out_amt"]].rename(columns={"out_amt": "out_next"})
    nxt["day"] = nxt["day"] - 1                      # wyplyw dnia d+1 przypiety do dnia d
    ad = ad.merge(nxt, on=["acc", "day"], how="left")
    del nxt
    ad["out_next"] = ad["out_next"].fillna(0.0)
    ad.loc[((ad["day"] + 1) // 7) != ad["week"], "out_next"] = 0.0   # tylko w obrebie tygodnia
    ad["fwd0"] = np.minimum(ad["in_amt"], ad["out_amt"])
    ad["fwd1"] = np.minimum(ad["in_amt"], ad["out_amt"] + ad["out_next"])
    ad["n_day"] = ad["in_n"] + ad["out_n"]
    ad["amt_day"] = ad["in_amt"] + ad["out_amt"]
    aw = (ad.groupby(["acc", "week"], sort=False)
            .agg(in_sum=("in_amt", "sum"), fwd0=("fwd0", "sum"), fwd1=("fwd1", "sum"),
                 max_day_n=("n_day", "max"), n_tot=("n_day", "sum"), amt_tot=("amt_day", "sum"))
            .reset_index())
    del ad
    safe_in = aw["in_sum"].where(aw["in_sum"] > 0, 1.0)
    aw["fwd_same"] = np.where(aw["in_sum"] > 0, aw["fwd0"] / safe_in, 0.0).clip(0, 1)
    aw["fwd_1d"] = np.where(aw["in_sum"] > 0, aw["fwd1"] / safe_in, 0.0).clip(0, 1)
    log(f"  konto x tydzien (wszystkie konta): {len(aw):,}")

    # --- 2. stopnie tygodniowe ---
    E = tx[["s", "r", "week"]].drop_duplicates()
    odeg = E.groupby(["s", "week"]).size().rename("out_deg").reset_index().rename(columns={"s": "acc"})
    ideg = E.groupby(["r", "week"]).size().rename("in_deg").reset_index().rename(columns={"r": "acc"})
    aw = aw.merge(odeg, on=["acc", "week"], how="left").merge(ideg, on=["acc", "week"], how="left")
    aw[["out_deg", "in_deg"]] = aw[["out_deg", "in_deg"]].fillna(0)
    log(f"  krawedzie tygodniowe: {len(E):,}")

    # --- 3. sasiedzi drugiego rzedu (co robia kontrahenci konta) ---
    nb = aw[["acc", "week", "fwd_1d", "in_deg", "out_deg"]].copy()
    nb["lin"] = np.log1p(nb["in_deg"]); nb["lout"] = np.log1p(nb["out_deg"])
    ein = E.merge(tgt.rename(columns={"acc": "r"}), on=["r", "week"])           # y -> A
    eout = E.merge(tgt.rename(columns={"acc": "s"}), on=["s", "week"])          # A -> x
    f_in = (ein.merge(nb.rename(columns={"acc": "s"}), on=["s", "week"], how="left")
               .groupby(["r", "week"]).agg(in_nb_fwd=("fwd_1d", "mean"), in_nb_indeg=("lin", "mean"))
               .reset_index().rename(columns={"r": "acc"}))
    f_out = (eout.merge(nb.rename(columns={"acc": "r"}), on=["r", "week"], how="left")
                .groupby(["s", "week"]).agg(out_nb_fwd=("fwd_1d", "mean"), out_nb_outdeg=("lout", "mean"))
                .reset_index().rename(columns={"s": "acc"}))

    # --- 4. cykle: A->x->A (cyc2), A->x->y->A (cyc3), x z out-degree <= cap ---
    P1 = eout.rename(columns={"s": "acc", "r": "x"})[["acc", "week", "x"]]
    P2 = ein.rename(columns={"r": "acc", "s": "y"})[["acc", "week", "y"]]
    c2 = P1.merge(P2, left_on=["acc", "week", "x"], right_on=["acc", "week", "y"])[["acc", "week"]]
    c2 = c2.drop_duplicates().assign(cyc2=1.0)
    small = odeg[odeg["out_deg"] <= cap].rename(columns={"acc": "s"})[["s", "week"]]
    Esm = E.merge(small, on=["s", "week"]).rename(columns={"s": "x", "r": "y"})
    J = P1.merge(Esm, on=["x", "week"])
    J = J[(J["y"] != J["acc"]) & (J["x"] != J["acc"])]
    c3 = J.merge(P2, on=["acc", "week", "y"])[["acc", "week"]].drop_duplicates().assign(cyc3=1.0)
    log(f"  cykle: cyc2 okien {len(c2):,}, cyc3 okien {len(c3):,} (zlaczenie J: {len(J):,} wierszy)")
    del J, Esm, P1, P2, ein, eout

    # --- 5. historia konta (tylko tygodnie wczesniejsze) ---
    awt = aw[aw["acc"].isin(tgt_acc)][["acc", "week", "n_tot", "amt_tot", "max_day_n"]].copy()
    awt["amt_mean"] = awt["amt_tot"] / awt["n_tot"].where(awt["n_tot"] > 0, 1)
    first_w = awt.groupby("acc")["week"].min().rename("first_w").reset_index()
    h = tgt.merge(awt[["acc", "week", "n_tot", "amt_mean", "max_day_n"]], on=["acc", "week"], how="left")
    prev_n, prev_amt, act = np.zeros(len(h)), np.zeros(len(h)), np.zeros(len(h))
    for lag in (1, 2, 3, 4):
        sh = awt[["acc", "week", "n_tot", "amt_mean"]].copy(); sh["week"] = sh["week"] + lag
        m = h[["acc", "week"]].merge(sh, on=["acc", "week"], how="left")
        ok = m["n_tot"].notna().to_numpy()
        act += ok
        prev_n += np.where(ok, m["n_tot"].fillna(0).to_numpy(), 0)
        prev_amt += np.where(ok, m["amt_mean"].fillna(0).to_numpy(), 0)
    has = act > 0
    mean_n = np.where(has, prev_n / np.maximum(act, 1), 0)
    mean_amt = np.where(has, prev_amt / np.maximum(act, 1), 0)
    h["hist_active4"] = act
    h["hist_n_lr"] = np.where(has, np.log1p(h["n_tot"].fillna(0)) - np.log1p(mean_n), 0.0)
    h["hist_amt_lr"] = np.where(has, np.log1p(h["amt_mean"].fillna(0)) - np.log1p(mean_amt), 0.0)
    h = h.merge(first_w, on="acc", how="left")
    h["acct_age"] = np.log1p((h["week"] - h["first_w"]).clip(lower=0))
    h["max_day_share"] = h["max_day_n"] / h["n_tot"].where(h["n_tot"] > 0, 1)

    # --- 6. udzial nowych kontrahentow (pierwszy raz widziani w tym tygodniu) ---
    sa = tx.loc[tx["s"].isin(tgt_acc), ["s", "r", "week"]].rename(columns={"s": "acc", "r": "cp"})
    ra = tx.loc[tx["r"].isin(tgt_acc), ["r", "s", "week"]].rename(columns={"r": "acc", "s": "cp"})
    pairs = pd.concat([sa, ra], ignore_index=True).drop_duplicates()
    del sa, ra
    fw = pairs.groupby(["acc", "cp"])["week"].min().rename("fw").reset_index()
    pw = pairs.merge(tgt, on=["acc", "week"]).merge(fw, on=["acc", "cp"])
    pw["new"] = (pw["fw"] == pw["week"]).astype("float64")
    ncp = pw.groupby(["acc", "week"])["new"].mean().rename("new_cp_share").reset_index()
    del pairs, fw, pw

    # --- 7. bilans z kolumn v0 ---
    b = win[["acc", "week", "n", "n_out", "amt_in_sum", "amt_out_sum"]].drop_duplicates(["acc", "week"])
    tot = b["amt_in_sum"] + b["amt_out_sum"]
    b = b.assign(bal_exact=np.where(tot > 0, 1 - (b["amt_in_sum"] - b["amt_out_sum"]).abs() / tot.where(tot > 0, 1), 0.0),
                 n_bal=(((b["n"] - b["n_out"]) == b["n_out"]) & (b["n_out"] > 0)).astype("float64"))

    out = (tgt.merge(aw[["acc", "week", "fwd_same", "fwd_1d"]], on=["acc", "week"], how="left")
              .merge(b[["acc", "week", "bal_exact", "n_bal", "n"]], on=["acc", "week"], how="left")
              .merge(f_in, on=["acc", "week"], how="left").merge(f_out, on=["acc", "week"], how="left")
              .merge(c2, on=["acc", "week"], how="left").merge(c3, on=["acc", "week"], how="left")
              .merge(h[["acc", "week", "n_tot", "hist_active4", "hist_n_lr", "hist_amt_lr", "acct_age",
                        "max_day_n", "max_day_share"]], on=["acc", "week"], how="left")
              .merge(ncp, on=["acc", "week"], how="left"))
    # zgodnosc definicji dnia/tygodnia z freeze v0: liczba transakcji okna musi sie zgadzac
    match = float((out["n_tot"] == out["n"]).mean())
    log(f"  zgodnosc n z freeze v0: {match:.4%}")
    if match < 0.999:
        raise ValueError(f"definicja tygodnia rozjechana z freeze v0 (zgodnosc {match:.2%})")
    out[NEW] = out[NEW].fillna(0.0).astype("float32")
    return out[["acc", "week"] + NEW]

In [ ]:
if FEAT_CACHE.exists():
    feats = pd.read_parquet(FEAT_CACHE)
    print("cechy v1 z cache:", FEAT_CACHE)
else:
    t0 = time.time()
    raw = pd.read_csv(SRC, usecols=["Date", "Sender_account", "Receiver_account", "Amount"],
                      dtype={"Sender_account": "int64", "Receiver_account": "int64", "Amount": "float32"})
    d = pd.to_datetime(raw["Date"])
    tx = pd.DataFrame({"s": raw["Sender_account"].to_numpy(), "r": raw["Receiver_account"].to_numpy(),
                       "a": raw["Amount"].to_numpy(),
                       "day": (d - d.min()).dt.days.astype("int32").to_numpy()})   # ta sama definicja co freeze v0
    del raw, d
    tx["week"] = (tx["day"] // 7).astype("int32")
    print(f"transakcji: {len(tx):,}")
    feats = build_v1_features(tx, bundle["windows"], cap=30)
    del tx
    feats.to_parquet(FEAT_CACHE, index=False)
    print(f"cechy v1 policzone w {time.time() - t0:.0f} s -> {FEAT_CACHE}")
print(feats[NEW].describe().T[["mean", "std", "min", "max"]].round(3).to_string())

## Ewaluacja v0 vs v1 na pulach walidacyjnych

In [ ]:
WEAK = ["Cycle", "Layered_Fan_In", "Bipartite", "Stacked Bipartite",
        "Behavioural_Change_1", "Behavioural_Change_2"]
ECFG = SimpleNamespace(n_way=5, k_shot=5, query=5)
DEV = torch.device("cpu")


def standardize(W, F, log_cols, train_classes, labels):
    X = W[F].astype("float64").to_numpy().copy()
    heavy = [j for j, c in enumerate(F) if c in log_cols]
    X[:, heavy] = np.log1p(X[:, heavy])
    m = np.isin(labels, list(train_classes))          # statystyki tylko z klas treningowych folda
    mu, sd = X[m].mean(0), X[m].std(0)
    sd[sd == 0] = 1.0
    return ((X - mu) / sd).astype("float32")


def episode_idx(s):
    """Indeksy wierszy epizodu (support 5 + query 5 na klase, konta rozlaczne) + nazwy klas."""
    pick = s.rng.choice(len(s.classes), s.n_way, replace=False)
    rows = []
    for ci in pick:
        info = s.by_class[s.classes[ci]]
        sa, qa = s._split_accounts(info["acct_rows"])
        sp = np.concatenate([info["acct_rows"][a] for a in sa])
        qp = np.concatenate([info["acct_rows"][a] for a in qa])
        rows += [s.rng.choice(sp, s.k_shot, replace=False), s.rng.choice(qp, s.query, replace=False)]
    return np.concatenate(rows), [s.classes[ci] for ci in pick]


def _tally(store, pred, names):
    for p, t in zip(pred, np.repeat(np.arange(5), 5)):
        store[names[t]][0] += int(p == t)
        store[names[t]][1] += 1


def evaluate(W, feature_sets, log_cols, folds, fold_ids=(1, 2, 3), seeds=(0, 1, 2),
             iters=600, n_eval=300, rf_trees=100, log=print):
    labels = W["label"].to_numpy().astype(str)
    groups = W["acc"].to_numpy()
    res = defaultdict(lambda: defaultdict(lambda: [0, 0]))      # (metoda, cechy) -> klasa -> [trafienia, razem]
    for k in fold_ids:
        spec = folds[k]
        dummy = np.zeros((len(W), 1), dtype="float32")
        vs = GroupedEpisodeSampler(dummy, labels, groups, spec["val"], ECFG, "val",
                                   np.random.default_rng(1000 + k))
        episodes = [episode_idx(vs) for _ in range(n_eval)]     # WSPOLNE dla v0 i v1
        for fs, F in feature_sets.items():
            X = standardize(W, F, log_cols, spec["train"], labels)
            y = np.repeat(np.arange(5), 5)
            for idx, names in episodes:
                x = X[idx].reshape(5, 10, -1)
                xs, xq = x[:, :5].reshape(25, -1), x[:, 5:].reshape(25, -1)
                protos = x[:, :5].mean(1)
                _tally(res[("nc_raw", fs)], ((xq[:, None, :] - protos[None]) ** 2).sum(-1).argmin(1), names)
                rf = RandomForestClassifier(n_estimators=rf_trees, random_state=0).fit(xs, y)
                _tally(res[("rf", fs)], rf.predict(xq), names)
            for seed in seeds:
                tr = GroupedEpisodeSampler(X, labels, groups, spec["train"], ECFG, "train",
                                           np.random.default_rng(seed))   # ten sam przebieg epizodow dla v0 i v1
                torch.manual_seed(seed)
                m = MLP(in_features=X.shape[1], hid=64)
                opt = torch.optim.Adam(m.parameters(), lr=1e-3)
                for _ in range(iters):
                    loss, _ = fast_adapt(m, tr.sample(), 5, 5, 5, DEV)
                    opt.zero_grad(); loss.backward(); opt.step()
                m.eval()
                with torch.no_grad():
                    for idx, names in episodes:
                        emb = m(torch.from_numpy(X[idx])).reshape(5, 10, -1)
                        pred = torch.cdist(emb[:, 5:].reshape(25, -1), emb[:, :5].mean(1)).argmin(1).numpy()
                        _tally(res[("protonet", fs)], pred, names)
            log(f"  fold {k} | cechy {fs} ({len(F)}) gotowe")
    return res


def report(res, weak=WEAK, log=print):
    rec = lambda key, c: res[key][c][0] / res[key][c][1] if res[key][c][1] else float("nan")
    cols = [("protonet", "v0"), ("protonet", "v1"), ("nc_raw", "v0"), ("nc_raw", "v1"), ("rf", "v0"), ("rf", "v1")]
    classes = sorted(res[cols[0]].keys())
    log(f"{'klasa':22s} " + " ".join(f"{m[:5]}-{f}".rjust(10) for m, f in cols))
    for c in classes:
        log(f"{c:22s} " + " ".join(f"{rec(k, c):10.3f}" for k in cols) + ("  *" if c in weak else ""))
    wk = {k: float(np.nanmean([rec(k, c) for c in weak if c in classes])) for k in cols}
    log("\nsredni recall slabych klas (*): " + " | ".join(f"{m}-{f} {wk[(m, f)]:.3f}" for m, f in cols))
    return wk

In [ ]:
W = bundle["windows"].merge(feats, on=["acc", "week"], how="left", validate="one_to_one")
assert len(W) == len(bundle["windows"]) and not W[NEW].isna().any().any(), "brak cech v1 dla czesci okien"
F0 = bundle["schema"]["feature_names"]
F1 = F0 + NEW
log_cols = set(bundle["schema"]["log1p_features"]) | {c for c in NEW if W[c].min() >= 0 and W[c].max() > 50}
print("log1p dla nowych cech:", sorted(log_cols & set(NEW)))

t0 = time.time()
res = evaluate(W, {"v0": F0, "v1": F1}, log_cols, bundle["folds"], fold_ids=(1, 2, 3),
               seeds=(0, 1, 2), iters=600, n_eval=300, rf_trees=100)
print(f"ewaluacja: {time.time() - t0:.0f} s\n")
missing = [c for c in WEAK if res[("protonet", "v0")][c][1] == 0]
if missing:
    raise RuntimeError(f"slabe klasy nie trafily do ewaluacji: {missing} - sprawdz fold_ids i pule walidacyjne")
wk = report(res)

d_proto = wk[("protonet", "v1")] - wk[("protonet", "v0")]
d_info = max(wk[("nc_raw", "v1")] - wk[("nc_raw", "v0")], wk[("rf", "v1")] - wk[("rf", "v0")])
print(f"\nDelta ProtoNet (v1 - v0): {d_proto:+.3f} | Delta najlepszego ucznia bez metryki: {d_info:+.3f}")
if d_proto >= 0.10:
    verdict = "PASS: freeze samld-v1 + kampania"
elif d_info >= 0.10:
    verdict = "INFORMACJA JEST, ProtoNet ja tlumi: poprawka modelu (dystans hybrydowy), nie GNN"
elif max(d_proto, d_info) >= 0.05:
    verdict = "CZESCIOWO: informacja niewyczerpana recznymi cechami -> uzasadnienie dla GNN"
else:
    verdict = "BRAK WZROSTU: sygnatura nie nalezy do konta -> klasyfikacja przypadkow"
print("WERDYKT:", verdict)

RESULTS.write_text(json.dumps({
    "v0_sha": sha_v0, "new_features": NEW, "verdict": verdict,
    "weak_recall": {f"{m}-{f}": v for (m, f), v in wk.items()},
    "per_class": {f"{m}-{f}": {c: res[(m, f)][c][0] / res[(m, f)][c][1] for c in res[(m, f)]}
                  for (m, f) in res},
}, indent=2, ensure_ascii=False))
print("wyniki zapisane:", RESULTS)